In [1]:
import torch
import torch.nn as nn
import math

class PositionEmbedding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[: x.size(1), :].unsqueeze(0).to(x.device)
        return x


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.dropout = nn.Dropout(dropout)

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)

        Q = self.Wq(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.Wk(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.Wv(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-1, -2)) / math.sqrt(self.d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))

        attn_weights = self.dropout(torch.softmax(scores, dim=-1))

        output = (
            torch.matmul(attn_weights, V)
            .transpose(1, 2)
            .contiguous()
            .view(batch_size, -1, self.d_k * self.num_heads)
        )

        output = self.Wo(output)
        return output, attn_weights


class FFN(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.activation = nn.ReLU()

    def forward(self, x):
        output = self.linear2(self.activation(self.linear1(x)))
        return output


class EncodeLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.MultiHeadAttention = MultiHeadAttention(d_model, num_heads, dropout)
        self.FFN = FFN(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        attn_output, attn_weights = self.MultiHeadAttention(x, x, x, src_mask)

        x = self.norm1(x + self.dropout(attn_output))
        x = self.norm2(x + self.dropout(self.FFN(x)))
        return x


class DecodeLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.FFN = FFN(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        attn1, _ = self.self_attn(x, x, x, mask=tgt_mask)
        x = self.norm1(x + self.dropout(attn1))

        attn2, _ = self.cross_attn(x, enc_output, enc_output, mask=src_mask)
        x = self.norm2(x + self.dropout(attn2))

        ffn_out = self.FFN(x)

        x = self.norm3(x + self.dropout(ffn_out))

        return x


class Encoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [EncodeLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )

        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, src_mask=None):
        for layer in self.layers:
            x = layer(x, src_mask)

        return self.norm(x)


class Decoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [DecodeLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, enc_output, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)

        return self.norm(x)


class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=512,
        num_heads=8,
        num_layers=6,
        d_ff=2048,
        max_len=5000,
        dropout=0.1,
    ):

        super().__init__()

        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)

        self.positional_encoding = PositionEmbedding(d_model, max_len)

        self.Encoder = Encoder(num_layers, d_model, num_heads, d_ff, dropout)
        self.Decoder = Decoder(num_layers, d_model, num_heads, d_ff, dropout)

        self.generator = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.d_model = d_model

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        src_emb = self.dropout(
            self.positional_encoding(self.src_embedding(src) * math.sqrt(self.d_model))
        )

        tgt_emb = self.dropout(
            self.positional_encoding(self.tgt_embedding(tgt) * math.sqrt(self.d_model))
        )

        enc_output = self.Encoder(src_emb, src_mask)

        dec_output = self.Decoder(tgt_emb, enc_output, src_mask, tgt_mask)

        output = self.generator(dec_output)

        return output


In [ ]:


# ==========================================
# 1. 设置假数据的超参数
# ==========================================
batch_size = 2  # 批次大小
src_seq_len = 10  # 源语言序列长度 (比如一句话有10个词)
tgt_seq_len = 12  # 目标语言序列长度 (比如翻译出的句子有12个词)
src_vocab_size = 5000  # 源语言词表大小
tgt_vocab_size = 5000  # 目标语言词表大小
d_model = 512  # 模型的隐藏层维度

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ==========================================
# 2. 实例化你写的 Transformer 模型
# ==========================================
model = Transformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=d_model,
    num_heads=8,
    num_layers=6,
    d_ff=2048,
    max_len=100,  # 测试时位置编码长度设小一点即可
    dropout=0.1,
).to(device)

# 切换到评估模式 (这会关闭 Dropout，让多次运行输出一致)
model.eval()

# ==========================================
# 3. 生成 Dummy Data (假数据)
# ==========================================
# 注意：输入给 Embedding 层的必须是 LongTensor (整数索引)
# 我们生成随机的词汇 ID 来模拟真实的输入 token
src = torch.randint(0, src_vocab_size, (batch_size, src_seq_len)).to(device)
tgt = torch.randint(0, tgt_vocab_size, (batch_size, tgt_seq_len)).to(device)


# ==========================================
# 4. 生成 Look-ahead Mask (掩盖目标序列的未来信息)
# ==========================================
# 在训练 Decoder 时，生成第 t 个词不能看到 t 之后的词。
# 你的 MultiHeadAttention 里写的是：mask == 0 时填入 -inf
# 所以我们需要一个下三角全为 1，上三角全为 0 的矩阵
def create_look_ahead_mask(size):
    # torch.tril 会保留矩阵的下三角部分，其余置 0
    mask = torch.tril(torch.ones(size, size))
    # 增加两个维度，变成 [1, 1, size, size]，以便和 MultiHeadAttention 里的 scores 广播匹配
    return mask.unsqueeze(0).unsqueeze(0).to(device)


tgt_mask = create_look_ahead_mask(tgt_seq_len)
# 简单起见，这里假设没有 Padding（填充），所以源掩码为空
src_mask = None

# ==========================================
# 5. 运行模型 (Forward Pass)
# ==========================================
with torch.no_grad():  # 测试阶段不需要计算梯度
    output = model(src, tgt, src_mask=src_mask, tgt_mask=tgt_mask)

# ==========================================
# 6. 打印结果，验证维度
# ==========================================
print("\n--- 维度验证 ---")
print(f"输入 src 的形状: {src.shape}  -> [batch_size, src_seq_len]")
print(f"输入 tgt 的形状: {tgt.shape}  -> [batch_size, tgt_seq_len]")
print(f"目标 Mask 的形状: {tgt_mask.shape} -> [1, 1, tgt_seq_len, tgt_seq_len]")
print("-" * 20)
print(f"⭐ 模型输出的形状: {output.shape}")
print(
    f"期望输出的形状: torch.Size([{batch_size}, {tgt_seq_len}, {tgt_vocab_size}]) -> [batch_size, tgt_seq_len, tgt_vocab_size]"
)


Using device: cuda

--- 维度验证 ---
输入 src 的形状: torch.Size([2, 10])  -> [batch_size, src_seq_len]
输入 tgt 的形状: torch.Size([2, 12])  -> [batch_size, tgt_seq_len]
目标 Mask 的形状: torch.Size([1, 1, 12, 12]) -> [1, 1, tgt_seq_len, tgt_seq_len]
--------------------
⭐ 模型输出的形状: torch.Size([2, 12, 5000])
期望输出的形状: torch.Size([2, 12, 5000]) -> [batch_size, tgt_seq_len, tgt_vocab_size]
